# Fertige AL-RNN-Runs auf konstanten Parametertrajektorien testen

Kein Training und keine neue Simulation. Pro Parameterwert wird eine gespeicherte Trajektorie ausgewaehlt. Ab `TEST_START` erhaelt das Modell genau einen beobachteten Startzustand und danach nur den externen Parameterinput. Ground Truth und Vorhersage werden in physikalischen Koordinaten dargestellt.

Der voreingestellte Dadras-Run liegt auf dem Server. Dort dieses Notebook mit dem b-tipping-venv als Kernel ausfuehren; lokal muss der Run zuvor vorhanden sein. Die benoetigten Trainingsstatistiken werden aus dem damaligen Trainingsdatenordner gelesen.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'alrnn_python',
              Path.home() / 'ALRNN-DSR-extended/alrnn_python',
              Path(r'C:\Users\koray\OneDrive\Dokumente\Studium\Master\Heidelberg\Master_Thesis\ALRNN-DSR-extended\alrnn_python')]
ALRNN_DIR = next((p.resolve() for p in candidates if (p / 'alrnn_model.py').is_file()), None)
if ALRNN_DIR is None:
    raise FileNotFoundError('Notebook im extended Repository oeffnen oder ALRNN_DIR explizit setzen.')
sys.path.insert(0, str(ALRNN_DIR))
print('AL-RNN code:', ALRNN_DIR)

## Einstellungen
`TRAJECTORY_PER_PARAMETER=0` waehlt die erste Trajektorie je Parameter, `1` die zweite usw. `TEST_START=80000` waehlt den letzten 20%-Abschnitt der vorhandenen 100000-Punkte-Trajektorien. Das ist eine Auswertungsauswahl, nicht die Behauptung, dass dieses Modell auf den ersten 80% dieses Testdatensatzes trainiert wurde.

Die erste vorhergesagte Probe wird korrekt mit `TEST_START + 1` verglichen. `DISCARD_STEPS=0` verwirft keine Vorhersagen. `TRAINING_DATA_DIR=None` sucht die Trainingsstatistiken automatisch; nach einem Umzug hier den passenden Ordner angeben.

In [ ]:
RUN_DIR = ALRNN_DIR / 'runs/term_ablations/dadras/combined_dadras_balanced/20261102_001412_188043_1162805/cphi/dadras_e_5p9_to_6p2'
if os.name == 'nt':
    DATA_DIR = ALRNN_DIR.parents[1] / 'generate_trainingdata/diffrentbutconst_prams/single_new_regimes_const/dadras_e_49_to_64/trajectories/values_8'
else:
    DATA_DIR = Path('/export/home/klenkeit/training_data/single_new_regimes_const/dadras_e_49_to_64/values_8')
TRAINING_DATA_DIR = None
DEVICE = 'auto'  # 'auto', 'cuda', or 'cpu'
GPU_ID = 0
CPU_THREADS = 4
TRAJECTORY_PER_PARAMETER = 0
TEST_START = 80000
FORECAST_STEPS = 10000
DISCARD_STEPS = 0
SAVE_RESULTS = False
OUTPUT_DIR = None  # Default: RUN_DIR / trajectory_tests / <timestamp>

In [ ]:
for name in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[name] = str(CPU_THREADS)
import hashlib
import json
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
import torch
from alrnn_model import load_saved_model
from compare_model_bifurcation_diagrams import load_training_coordinates

RUN_DIR, DATA_DIR = Path(RUN_DIR).expanduser().resolve(), Path(DATA_DIR).expanduser().resolve()
for path in [RUN_DIR / 'model.pt', RUN_DIR / 'metrics.json', DATA_DIR / 'metadata.json',
             DATA_DIR / 'trajectories.npy', DATA_DIR / 'trajectory_phi_constants.npy']:
    if not path.is_file():
        raise FileNotFoundError(f'{path} fehlt. Server-Kernel verwenden oder Pfade anpassen.')
if DEVICE not in ('auto', 'cpu', 'cuda'):
    raise ValueError('DEVICE muss auto, cpu oder cuda sein.')
if CPU_THREADS < 1 or TEST_START < 0 or FORECAST_STEPS < 1 or DISCARD_STEPS < 0 or TRAJECTORY_PER_PARAMETER < 0:
    raise ValueError('Ungueltige Thread-, Zeit- oder Trajektorieneinstellung.')
torch.set_num_threads(CPU_THREADS)
use_cuda = DEVICE == 'cuda' or (DEVICE == 'auto' and torch.cuda.is_available())
if use_cuda:
    if not torch.cuda.is_available() or not 0 <= GPU_ID < torch.cuda.device_count():
        raise ValueError(f'CUDA GPU {GPU_ID} ist nicht verfuegbar.')
    torch.cuda.set_device(GPU_ID)
device = torch.device(f'cuda:{GPU_ID}' if use_cuda else 'cpu')
model, run_metrics, checkpoint = load_saved_model(RUN_DIR, device=device)
if model.N != 3 or model.phi_dim not in (0, 1):
    raise ValueError('Dieses Notebook erwartet drei beobachtete Dimensionen und optional skalares phi.')
print('Device:', device)
print('Checkpoint:', checkpoint)
print('Model:', model.model_config())

## Koordinaten und Parameterbezug
Der Testdatensatz ist pro Trajektorie standardisiert. Die gespeicherten Mittelwerte und Standardabweichungen invertieren diese Transformation. Fuer global trainierte Modelle werden anschliessend deren feste Trainingsstatistiken verwendet. Bei aelteren, pro Trajektorie trainierten Modellen bleibt die jeweilige Testtrajektorienskalierung das Evaluationsprotokoll; diese verwendet Ground-Truth-Statistiken. Auch phi wird aus dem physikalischen Parameter in den Bezug des Trainingsdatensatzes umgerechnet.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def phi_reference(metadata):
    if 'phi_reference' in metadata:
        reference = metadata['phi_reference']
        boundary, scale = float(reference['boundary']), float(reference['scale'])
    else:
        regimes = metadata.get('regimes', [])
        if len(regimes) != 1:
            raise ValueError('Ein eindeutiger DS-Parameterbezug wird benoetigt.')
        regime = regimes[0]
        boundary = float(regime['boundary'])
        scale = max(abs(float(regime['start']) - boundary), abs(float(regime['end']) - boundary))
    if not np.isfinite([boundary, scale]).all() or scale <= 0:
        raise ValueError('Ungueltiger phi-Bezug.')
    return boundary, scale

def resolve_training_data():
    if TRAINING_DATA_DIR is not None:
        candidates = [Path(TRAINING_DATA_DIR)]
    else:
        saved = run_metrics.get('args', {}).get('data_dir')
        candidates = [Path(saved.replace(chr(92), '/'))] if saved else []
        candidates += [ALRNN_DIR.parents[1] / 'generate_trainingdata/mixed_parameter_trajectories/datasets/combined_dadras' / RUN_DIR.name,
                       Path.home() / 'training_data/mixed_parameter_trajectories/dadras/combined_dadras' / RUN_DIR.name]
    for candidate in candidates:
        candidate = candidate.expanduser()
        if (candidate / 'metadata.json').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('Trainingsdaten-Metadaten fehlen. TRAINING_DATA_DIR auf den damaligen Trainingsdatenordner setzen.')

data_metadata = read_json(DATA_DIR / 'metadata.json')
training_data_dir = resolve_training_data()
training_metadata = read_json(training_data_dir / 'metadata.json')
source_boundary, source_scale = phi_reference(data_metadata)
training_boundary, training_scale = phi_reference(training_metadata)
source_regimes = [item['name'] for item in data_metadata.get('regimes', [])]
training_regimes = [item['name'] for item in training_metadata.get('regimes', [])]
if source_regimes != training_regimes:
    raise ValueError(f'DS stimmen nicht ueberein: Test={source_regimes}, Training={training_regimes}')
normalization = training_metadata.get('config', {}).get('standardization')
if normalization == 'global':
    coordinates = load_training_coordinates(training_data_dir)
    training_mean, training_std = coordinates.mean, coordinates.std
elif normalization == 'per_trajectory' or (normalization is None and 'args' in training_metadata):
    normalization = 'per_trajectory'
    training_mean = training_std = None
elif normalization == 'none':
    training_mean, training_std = np.zeros(3), np.ones(3)
else:
    raise ValueError(f'Nicht unterstuetzte Trainingsnormalisierung: {normalization}')
print('Trainingsstatistiken:', training_data_dir)
print('Trainingsnormalisierung:', normalization)
print(f'Modell-phi = (raw - {training_boundary}) / {training_scale}')

In [ ]:
trajectories = np.load(DATA_DIR / 'trajectories.npy', mmap_mode='r', allow_pickle=False)
phi_constants = np.load(DATA_DIR / 'trajectory_phi_constants.npy', allow_pickle=False).reshape(-1)
if trajectories.ndim != 3 or trajectories.shape[0] != len(phi_constants) or trajectories.shape[-1] != 3:
    raise ValueError('Erwartet wird ein konstanter Datensatz mit Form (Trajektorie, Zeit, 3).')
if not np.isfinite(phi_constants).all():
    raise ValueError('Nicht-endliche Parameterwerte im Testdatensatz.')
raw_constants_path = DATA_DIR / 'trajectory_phi_raw_constants.npy'
raw_constants = (np.load(raw_constants_path, allow_pickle=False).reshape(-1)
                 if raw_constants_path.is_file() else source_boundary + source_scale * phi_constants)
names_path = DATA_DIR / 'trajectory_regime_names.npy'
regime_names = (np.load(names_path, allow_pickle=False).astype(str) if names_path.is_file()
                else np.full(len(phi_constants), source_regimes[0]))
if raw_constants.shape != phi_constants.shape or regime_names.shape != phi_constants.shape or not np.isfinite(raw_constants).all():
    raise ValueError('Parameter- oder Regimezuordnung passt nicht zu den Trajektorien.')

def source_scaling(index):
    mean_path, std_path = DATA_DIR / 'trajectory_mean.npy', DATA_DIR / 'trajectory_std.npy'
    if mean_path.is_file() and std_path.is_file():
        mean = np.load(mean_path, mmap_mode='r', allow_pickle=False)[index, 0]
        std = np.load(std_path, mmap_mode='r', allow_pickle=False)[index, 0]
    else:
        value = float(phi_constants[index])
        label = f'{value:+.6g}'.replace('+', 'p').replace('-', 'm').replace('.', 'p')
        stem = f'{regime_names[index]}_phi_{label}'
        members = np.flatnonzero((regime_names == regime_names[index]) & np.isclose(phi_constants, value, atol=1e-5, rtol=0))
        row = int(np.flatnonzero(members == index)[0])
        mean_path, std_path = DATA_DIR / f'{stem}_mean.npy', DATA_DIR / f'{stem}_std.npy'
        if not mean_path.is_file() or not std_path.is_file():
            raise FileNotFoundError(f'Gespeicherte Normalisierungsstatistiken fehlen fuer {stem}.')
        mean = np.load(mean_path, allow_pickle=False)[row, 0]
        std = np.load(std_path, allow_pickle=False)[row, 0]
    mean, std = np.asarray(mean, dtype=np.float64), np.asarray(std, dtype=np.float64)
    if mean.shape != (3,) or std.shape != (3,) or not np.isfinite([mean, std]).all() or np.any(std <= 0):
        raise ValueError(f'Ungueltige Skalierung fuer Trajektorie {index}.')
    return mean, std

selected = []
for value in np.unique(phi_constants):
    members = np.flatnonzero(np.isclose(phi_constants, value, atol=1e-5, rtol=0))
    if TRAJECTORY_PER_PARAMETER >= len(members):
        raise ValueError(f'phi={value}: nur {len(members)} Trajektorien vorhanden.')
    index = int(members[TRAJECTORY_PER_PARAMETER])
    source_scaling(index)
    selected.append(index)
    print(f'phi={float(value):+.6f} | raw={float(raw_constants[index]):.6f} | trajectory={index}')
print('Datenform:', trajectories.shape, '| geladene Arrays sind memory-mapped')

## Freie Vorhersage
Nur der Startzustand initialisiert die latenten Koordinaten ueber das gespeicherte B; die ersten drei Koordinaten werden auf diesen beobachteten Zustand gesetzt. Danach gibt es kein Teacher Forcing. Ein Modell ohne externen Input erhaelt kein phi. Die Vorhersagelaenge wird an den verfuegbaren Testabschnitt angepasst. Nicht-endliche Rollouts werden markiert und im Plot nur bis zum ersten ungueltigen Punkt gezeigt.

In [ ]:
results = []
dt = float(data_metadata.get('args', {}).get('dt', data_metadata.get('config', {}).get('dt', 1.0)))
parameter_name = data_metadata['regimes'][0].get('parameter', 'parameter')
for index in selected:
    mean, std = source_scaling(index)
    available = trajectories.shape[1] - TEST_START - 1
    horizon = min(FORECAST_STEPS, available - DISCARD_STEPS)
    if horizon < 1:
        raise ValueError(f'Trajektorie {index}: kein Testabschnitt nach TEST_START und DISCARD_STEPS vorhanden.')
    total_steps = DISCARD_STEPS + horizon
    # Undo test normalization before applying the model's training coordinates.
    source = np.asarray(trajectories[index, TEST_START:TEST_START + total_steps + 1], dtype=np.float64)
    raw = source * std + mean
    if not np.isfinite(raw).all():
        raise ValueError(f'Nicht-endliche Ground Truth fuer Trajektorie {index}.')
    model_mean = mean if normalization == 'per_trajectory' else training_mean
    model_std = std if normalization == 'per_trajectory' else training_std
    x0 = torch.tensor(((raw[0] - model_mean) / model_std)[None], dtype=torch.float32, device=device)
    raw_parameter = float(raw_constants[index])
    model_phi = (raw_parameter - training_boundary) / training_scale
    phi_input = torch.tensor([[model_phi]], dtype=torch.float32, device=device) if model.phi_dim else None
    with torch.inference_mode():
        z = x0 @ model.B
        z[:, :model.N] = x0
        prediction = torch.empty((total_steps, model.N), device=device)
        for step in range(total_steps):
            z = model(z, phi_input)
            prediction[step] = z[0, :model.N]
        predicted = prediction.cpu().numpy().astype(np.float64) * model_std + model_mean
    predicted = predicted[DISCARD_STEPS:]
    truth = raw[1 + DISCARD_STEPS:]
    valid = np.isfinite(predicted).all(axis=1)
    valid_steps = int(np.flatnonzero(~valid)[0]) if not valid.all() else horizon
    result = dict(trajectory_index=index, source_phi=float(phi_constants[index]),
                  raw_parameter=raw_parameter, model_phi=float(model_phi), steps=horizon,
                  finite=bool(valid.all()), finite_prefix_steps=valid_steps,
                  first_truth_index=TEST_START + 1 + DISCARD_STEPS,
                  source_mean=mean.tolist(), source_std=std.tolist(),
                  model_mean=np.asarray(model_mean).tolist(), model_std=np.asarray(model_std).tolist(),
                  truth=truth, prediction=predicted)
    results.append(result)
    print(f'{parameter_name}={raw_parameter:.6f} | {horizon} forecast steps | finite={result["finite"]}')

In [ ]:
output_dir = (Path(OUTPUT_DIR).expanduser().resolve() if OUTPUT_DIR is not None
              else RUN_DIR / 'trajectory_tests' / f'{DATA_DIR.name}_{datetime.now():%Y%m%d_%H%M%S_%f}')
if SAVE_RESULTS:
    output_dir.mkdir(parents=True, exist_ok=True)
for result in results:
    truth, predicted = result['truth'], result['prediction'][:result['finite_prefix_steps']]
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(*truth.T, color='tab:blue', linewidth=0.7, alpha=0.65, label='Ground Truth')
    ax.plot(*predicted.T, color='tab:orange', linewidth=0.8, label='Free AL-RNN rollout')
    state = '' if result['finite'] else f' | WARNING: only {result["finite_prefix_steps"]} finite steps'
    ax.set_title(f'{parameter_name}={result["raw_parameter"]:.6f}, phi={result["source_phi"]:+.4f}' + chr(10) +
                 f'traj={result["trajectory_index"]}, steps={result["steps"]}{state}')
    ax.set(xlabel='x (raw)', ylabel='y (raw)', zlabel='z (raw)')
    ax.legend()
    fig.tight_layout()
    if SAVE_RESULTS:
        stem = f'trajectory_{result["trajectory_index"]}'
        fig.savefig(output_dir / f'{stem}_3d.png', dpi=180)
        np.savez_compressed(output_dir / f'{stem}.npz', truth=truth, prediction=result['prediction'],
                            raw_parameter=result['raw_parameter'], model_phi=result['model_phi'], dt=dt)
    plt.show()
    plt.close(fig)

if SAVE_RESULTS:
    digest = hashlib.sha256()
    with checkpoint.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    summary = dict(created_at_utc=datetime.now(timezone.utc).isoformat(), run_dir=str(RUN_DIR),
                   checkpoint=str(checkpoint), checkpoint_sha256=digest.hexdigest(),
                   model_config=model.model_config(), data_dir=str(DATA_DIR),
                   training_data_dir=str(training_data_dir), training_normalization=normalization,
                   test_start=TEST_START, requested_forecast_steps=FORECAST_STEPS, discard_steps=DISCARD_STEPS,
                   trajectory_per_parameter=TRAJECTORY_PER_PARAMETER, dt=dt,
                   evaluation=[{k: v for k, v in result.items() if k not in ('truth', 'prediction')} for result in results])
    (output_dir / 'evaluation.json').write_text(json.dumps(summary, indent=2) + chr(10), encoding='utf-8')
    print('Saved plots, arrays and provenance:', output_dir)